# Silver Data EDA

Senior data-engineering inspection notebook for the Silver pageview layer.

Goals:

- Confirm Silver matches the Bronze manifest and active 72-hour scope.
- Inspect Hive partition layout, Parquet file sizes, row counts, and schema.
- Surface quality issues before Gold modeling: missing partitions, null titles, negative metrics, duplicate keys, skew, and small-file pressure.
- Keep expensive scans optional and explicit.

This notebook is read-only.


## 1. Setup


In [ ]:
from __future__ import annotations

import json
import re
import sys
from pathlib import Path

import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from wikitrend.pageviews import DEFAULT_SOURCE_PROJECTS, PROJECT_CODE_MAP
from wikitrend.silver_validation import REQUIRED_COLUMNS, parse_silver_partition

SILVER_DIR = ROOT / "data" / "processed" / "silver" / "pageviews"
QUARANTINE_DIR = ROOT / "data" / "processed" / "quarantine" / "pageviews"
MANIFEST_PATH = ROOT / "data" / "raw" / "pageviews_manifest.json"
VALIDATION_REPORT_PATH = ROOT / "data" / "processed" / "validation" / "silver_pageviews_validation.json"

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8-sig"))


def bytes_to_gb(value: int | float) -> float:
    return round(float(value) / 1_000_000_000, 3)

assert SILVER_DIR.exists(), f"Missing Silver directory: {SILVER_DIR}"
assert MANIFEST_PATH.exists(), f"Missing Bronze manifest: {MANIFEST_PATH}"


## 2. Validation Report And Dataset Contract


In [ ]:
manifest = read_json(MANIFEST_PATH)
manifest_df = pd.DataFrame(manifest.get("files", []))
if not manifest_df.empty:
    manifest_df["timestamp_hour"] = pd.to_datetime(manifest_df["timestamp_hour"], utc=True)
    manifest_df["date"] = manifest_df["timestamp_hour"].dt.date.astype(str)
    manifest_df["hour"] = manifest_df["timestamp_hour"].dt.hour

validation_report = read_json(VALIDATION_REPORT_PATH) if VALIDATION_REPORT_PATH.exists() else {}
validation_metrics = validation_report.get("metrics", {})

contract = {
    "validation_status": validation_report.get("status"),
    "validation_errors": len(validation_report.get("errors", [])),
    "validation_warnings": len(validation_report.get("warnings", [])),
    "manifest_hours": len(manifest_df),
    "silver_rows_reported": validation_metrics.get("silver_rows"),
    "silver_size_gb_reported": bytes_to_gb(validation_metrics.get("silver_size_bytes", 0)),
    "partition_hours_reported": validation_metrics.get("partition_hours"),
    "partition_combinations_reported": validation_metrics.get("partition_combinations"),
}

pd.DataFrame([contract])


In [ ]:
if validation_report:
    display(pd.DataFrame({"errors": validation_report.get("errors", [])}))
    display(pd.DataFrame({"warnings": validation_report.get("warnings", [])}))
else:
    print("No validation report found. Run validate_silver_pageviews.py before relying on this notebook.")


## 3. Parquet And Partition Inventory


In [ ]:
parquet_files = sorted(SILVER_DIR.rglob("*.parquet"))
partition_rows = []

for path in parquet_files:
    partition = parse_silver_partition(path, SILVER_DIR)
    metadata = pq.ParquetFile(path).metadata
    row = {
        "path": path,
        "relative_path": path.relative_to(SILVER_DIR).as_posix(),
        "size_bytes": path.stat().st_size,
        "row_count": metadata.num_rows,
        "row_groups": metadata.num_row_groups,
    }
    if partition is not None:
        row.update(
            {
                "date": partition.date,
                "hour": partition.hour,
                "project": partition.project,
                "access_mode": partition.access_mode,
            }
        )
    partition_rows.append(row)

silver_file_df = pd.DataFrame(partition_rows)
if not silver_file_df.empty:
    silver_file_df["size_mb"] = silver_file_df["size_bytes"] / 1_000_000

inventory_summary = {
    "parquet_files": len(silver_file_df),
    "silver_size_gb": bytes_to_gb(silver_file_df["size_bytes"].sum()) if not silver_file_df.empty else 0.0,
    "footer_row_count": int(silver_file_df["row_count"].sum()) if not silver_file_df.empty else 0,
    "partition_hours": 0 if silver_file_df.empty else silver_file_df[["date", "hour"]].drop_duplicates().shape[0],
    "partition_combinations": 0 if silver_file_df.empty else silver_file_df[["date", "hour", "project", "access_mode"]].drop_duplicates().shape[0],
    "avg_file_mb": None if silver_file_df.empty else round(silver_file_df["size_mb"].mean(), 3),
    "p95_file_mb": None if silver_file_df.empty else round(silver_file_df["size_mb"].quantile(0.95), 3),
}

pd.DataFrame([inventory_summary])


In [ ]:
silver_file_df.head(20)


## 4. Coverage Matrix


In [ ]:
expected_hours = manifest_df[["date", "hour"]].drop_duplicates() if not manifest_df.empty else pd.DataFrame(columns=["date", "hour"])
expected_pairs = []
for source_project in DEFAULT_SOURCE_PROJECTS:
    dimensions = PROJECT_CODE_MAP[source_project]
    expected_pairs.append({"project": dimensions.project, "access_mode": dimensions.access_mode})
expected_pairs_df = pd.DataFrame(expected_pairs).drop_duplicates()

expected_grid = expected_hours.merge(expected_pairs_df, how="cross") if not expected_hours.empty else pd.DataFrame()
actual_grid = silver_file_df[["date", "hour", "project", "access_mode"]].drop_duplicates() if not silver_file_df.empty else pd.DataFrame(columns=["date", "hour", "project", "access_mode"])
coverage_df = expected_grid.merge(actual_grid, on=["date", "hour", "project", "access_mode"], how="outer", indicator=True)
coverage_df["coverage_status"] = coverage_df["_merge"].map({"both": "present", "left_only": "missing", "right_only": "extra"})

coverage_summary = coverage_df["coverage_status"].value_counts(dropna=False).rename_axis("coverage_status").reset_index(name="partitions")
coverage_summary


In [ ]:
coverage_df.loc[coverage_df["coverage_status"].ne("present")].head(100)


In [ ]:
if not coverage_df.empty:
    coverage_pivot = coverage_df.assign(project_access=coverage_df["project"] + ":" + coverage_df["access_mode"]).pivot_table(
        index=["date", "hour"], columns="project_access", values="coverage_status", aggfunc="first", fill_value="missing"
    )
    display(coverage_pivot.head(72))


## 5. Schema Contract


In [ ]:
silver_dataset = ds.dataset(SILVER_DIR, format="parquet", partitioning="hive")
schema_df = pd.DataFrame(
    [{"column": field.name, "type": str(field.type), "nullable": field.nullable} for field in silver_dataset.schema]
)
schema_df["required"] = schema_df["column"].isin(REQUIRED_COLUMNS)
missing_required_columns = sorted(REQUIRED_COLUMNS - set(schema_df["column"]))

display(schema_df)
print("Missing required columns:", missing_required_columns)


## 6. Row Count And File Layout Diagnostics


In [ ]:
if silver_file_df.empty:
    print("No Silver parquet files found.")
else:
    by_partition = (
        silver_file_df.groupby(["date", "hour", "project", "access_mode"], dropna=False)
        .agg(files=("path", "size"), rows=("row_count", "sum"), size_mb=("size_mb", "sum"), avg_file_mb=("size_mb", "mean"))
        .reset_index()
    )
    display(by_partition.sort_values("rows", ascending=False).head(50))


In [ ]:
if not silver_file_df.empty:
    small_files = silver_file_df.loc[silver_file_df["size_mb"].lt(16)]
    small_file_summary = {
        "small_files_under_16mb": len(small_files),
        "small_file_rate": 0 if len(silver_file_df) == 0 else len(small_files) / len(silver_file_df),
        "small_file_size_gb": bytes_to_gb(small_files["size_bytes"].sum()),
    }
    display(pd.DataFrame([small_file_summary]))
    display(small_files.sort_values("size_mb").head(50)[["relative_path", "row_count", "size_mb"]])


In [ ]:
if not silver_file_df.empty:
    hourly_rows = silver_file_df.groupby(["date", "hour"], as_index=False).agg(rows=("row_count", "sum"), size_mb=("size_mb", "sum"))
    display(hourly_rows.head())
    hourly_rows.assign(timestamp=pd.to_datetime(hourly_rows["date"]) + pd.to_timedelta(hourly_rows["hour"], unit="h")).plot(
        x="timestamp", y=["rows", "size_mb"], subplots=True, figsize=(14, 6), title="Silver rows and size by hour"
    )


## 7. Sample Rows And Top Pages


In [ ]:
SAMPLE_ROWS = 25_000
SAMPLE_COLUMNS = [
    "date",
    "hour",
    "project",
    "access_mode",
    "source_project",
    "language",
    "project_family",
    "page_title",
    "normalized_title",
    "view_count",
    "response_size",
    "source_filename",
]

sample_table = silver_dataset.head(SAMPLE_ROWS, columns=SAMPLE_COLUMNS)
sample_df = sample_table.to_pandas()

sample_df.head()


In [ ]:
if sample_df.empty:
    print("No sample rows loaded.")
else:
    display(
        sample_df.groupby(["project", "access_mode"], dropna=False)
        .agg(rows=("page_title", "size"), views=("view_count", "sum"), response_size=("response_size", "sum"))
        .assign(avg_response_size_per_view=lambda df: df["response_size"] / df["views"].where(df["views"].ne(0)))
        .sort_values("views", ascending=False)
    )


In [ ]:
if not sample_df.empty:
    display(
        sample_df.sort_values("view_count", ascending=False)[
            ["date", "hour", "project", "access_mode", "normalized_title", "view_count", "response_size", "source_filename"]
        ].head(50)
    )


## 8. Data Quality Checks


In [ ]:
quality_from_report = {
    "null_page_title_rows": validation_metrics.get("null_page_title_rows"),
    "null_normalized_title_rows": validation_metrics.get("null_normalized_title_rows"),
    "negative_view_count_rows": validation_metrics.get("negative_view_count_rows"),
    "negative_response_size_rows": validation_metrics.get("negative_response_size_rows"),
}

pd.DataFrame([quality_from_report])


In [ ]:
if sample_df.empty:
    print("No sample rows loaded.")
else:
    sample_quality = {
        "sample_rows": len(sample_df),
        "null_page_title_rows": int(sample_df["page_title"].isna().sum()),
        "null_normalized_title_rows": int(sample_df["normalized_title"].isna().sum()),
        "negative_view_count_rows": int(sample_df["view_count"].lt(0).sum()),
        "negative_response_size_rows": int(sample_df["response_size"].lt(0).sum()),
        "zero_view_rows": int(sample_df["view_count"].eq(0).sum()),
    }
    display(pd.DataFrame([sample_quality]))


## 9. Quarantine Inventory


In [ ]:
quarantine_files = sorted(QUARANTINE_DIR.rglob("*.parquet")) if QUARANTINE_DIR.exists() else []
quarantine_rows = []
for path in quarantine_files:
    metadata = pq.ParquetFile(path).metadata
    quarantine_rows.append(
        {
            "relative_path": path.relative_to(QUARANTINE_DIR).as_posix(),
            "size_bytes": path.stat().st_size,
            "row_count": metadata.num_rows,
        }
    )
quarantine_df = pd.DataFrame(quarantine_rows)

quarantine_summary = {
    "quarantine_dir_exists": QUARANTINE_DIR.exists(),
    "quarantine_files": len(quarantine_df),
    "quarantine_rows_from_footers": int(quarantine_df["row_count"].sum()) if not quarantine_df.empty else 0,
    "quarantine_size_gb": bytes_to_gb(quarantine_df["size_bytes"].sum()) if not quarantine_df.empty else 0.0,
}
pd.DataFrame([quarantine_summary])


In [ ]:
quarantine_df.head(50)


## 10. Optional Full Aggregates With DuckDB


In [ ]:
RUN_DUCKDB_AGGREGATES = False

if RUN_DUCKDB_AGGREGATES:
    import duckdb

    con = duckdb.connect()
    parquet_glob = str(SILVER_DIR / "**" / "*.parquet").replace("\\", "/")
    by_project = con.execute(
        f"""
        select
          date,
          hour,
          project,
          access_mode,
          count(*) as rows,
          sum(view_count) as views,
          sum(response_size) as response_size
        from read_parquet('{parquet_glob}', hive_partitioning=true)
        group by 1, 2, 3, 4
        order by date, hour, project, access_mode
        """
    ).df()
    display(by_project.head(50))
    display(by_project.groupby(["project", "access_mode"], as_index=False).agg(rows=("rows", "sum"), views=("views", "sum")).sort_values("views", ascending=False))
else:
    print("Set RUN_DUCKDB_AGGREGATES = True for full Silver metric aggregates.")


## 11. Optional Duplicate Natural Key Check


In [ ]:
CHECK_DUPLICATE_NATURAL_KEYS = False

if CHECK_DUPLICATE_NATURAL_KEYS:
    import duckdb

    con = duckdb.connect()
    parquet_glob = str(SILVER_DIR / "**" / "*.parquet").replace("\\", "/")
    duplicate_keys = con.execute(
        f"""
        select date, hour, source_project, page_title, count(*) as row_count
        from read_parquet('{parquet_glob}', hive_partitioning=true)
        group by 1, 2, 3, 4
        having count(*) > 1
        order by row_count desc
        limit 100
        """
    ).df()
    display(duplicate_keys)
else:
    print("Set CHECK_DUPLICATE_NATURAL_KEYS = True to run a full duplicate-key check.")


## 12. Silver Readiness Checklist

Silver is ready for Gold design when:

- Validation status is `pass`.
- Manifest hours equal Silver partition hours.
- Missing and extra partitions are zero.
- Required schema columns are present.
- Null title and negative metric counts are zero.
- File-size distribution is acceptable for local analysis. Many small files are fine for this research-scale dataset, but should be compacted before repeated production-style scans.
- Quarantine volume is understood.
- Full aggregates or duplicate-key checks have been run if Gold logic depends on exact uniqueness or totals.
